# NeuroVest — Signal History & Performance Demo

This notebook loads the latest signal store and shows:
1. Signal distribution over time
2. Hit-rate (realized PnL annotation)
3. Cumulative equity curve
4. Latest `outputs/metrics_*.json` summary

Run `python run_all.py --daily` first to populate `logs/labeled_predictions.csv`.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True

SIGNALS_CSV = Path('logs/labeled_predictions.csv')
OUTPUTS_DIR = Path('outputs')

## 1. Load Signal Store

In [ ]:
if not SIGNALS_CSV.exists():
    print(f'Signal store not found at {SIGNALS_CSV}.')
    print('Run:  python run_all.py --daily')
else:
    sig = pd.read_csv(SIGNALS_CSV, parse_dates=['Date'])
    sig = sig.sort_values('Date').reset_index(drop=True)
    print(f'Loaded {len(sig)} rows  ({sig["Date"].min().date()} → {sig["Date"].max().date()})')
    sig.tail(10)

## 2. Signal Distribution

In [ ]:
if 'sig' in dir():
    label_map = {0: 'CRASH', 1: 'NORMAL', 2: 'SPIKE'}
    counts = sig['Prediction'].map(label_map).value_counts()
    counts.plot(kind='bar', color=['red','grey','green'], title='Signal Distribution')
    plt.ylabel('Count')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
    print(counts)

## 3. Signal Probability Over Time

In [ ]:
if 'sig' in dir() and 'Proba' in sig.columns:
    fig, ax = plt.subplots()
    spikes = sig[sig['Prediction'] == 2]
    ax.plot(sig['Date'], sig['Proba'], label='p(long)', color='steelblue', linewidth=0.8)
    ax.scatter(spikes['Date'], spikes['Proba'], color='green', s=15, zorder=5, label='SPIKE signal')
    ax.axhline(0.45, color='orange', linestyle='--', linewidth=0.8, label='default threshold 0.45')
    ax.set_title('Predicted Probability Over Time')
    ax.set_ylabel('p(long=1)')
    ax.legend()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

## 4. Hit-Rate (Realized PnL)

In [ ]:
if 'sig' in dir() and 'realized_ret' in sig.columns:
    annotated = sig.dropna(subset=['realized_ret'])
    long_signals = annotated[annotated['Prediction'] == 2]
    if len(long_signals):
        hit_rate = (long_signals['realized_ret'] > 0).mean()
        avg_ret = long_signals['realized_ret'].mean()
        print(f'SPIKE signals: {len(long_signals)} annotated')
        print(f'Hit-rate:      {hit_rate:.1%}')
        print(f'Avg return:    {avg_ret:.3%}')

        long_signals['realized_ret'].hist(bins=30, color='steelblue', edgecolor='white')
        plt.axvline(0, color='red', linestyle='--')
        plt.title('Distribution of Realized Returns (SPIKE signals)')
        plt.xlabel('Return')
        plt.tight_layout()
        plt.show()
    else:
        print('No SPIKE signals with realized PnL yet. Run predict.py --fill-realized.')
else:
    print('No realized_ret column. Run: python predict.py --fill-realized')

## 5. Latest Backtest Metrics

In [ ]:
metrics_files = sorted(OUTPUTS_DIR.glob('metrics_*.json')) if OUTPUTS_DIR.exists() else []
if metrics_files:
    latest = metrics_files[-1]
    print(f'Loading {latest}')
    with open(latest) as f:
        m = json.load(f)

    display_keys = ['run_at', 'git_sha', 'trades', 'cagr', 'sharpe', 'sortino',
                    'max_drawdown', 'hit_rate', 'profit_factor', 'buy_hold_return', 'alpha']
    for k in display_keys:
        v = m.get(k, 'N/A')
        if isinstance(v, float):
            if k in ('cagr', 'max_drawdown', 'buy_hold_return', 'alpha', 'hit_rate'):
                print(f'  {k:>22}: {v:.2%}')
            else:
                print(f'  {k:>22}: {v:.3f}')
        else:
            print(f'  {k:>22}: {v}')
else:
    print(f'No metrics found in {OUTPUTS_DIR}. Run: python backtest.py')